In [33]:
import fiftyone as fo  

# 加载已存在的数据集  
dataset = fo.load_dataset("my-dataset")
session = fo.launch_app(dataset)

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port


### 在fiftyon__根据已有的boundingBox__应用__分类模型__二次过滤检测结果

In [34]:
import fiftyone as fo
from fiftyone import ViewField as F

dataset_name = "test_run2_image_copy"

dataset = fo.load_dataset(dataset_name)

# 假设 ds 为你的数据集，pred_field 为检测字段名（如 "pred_yolo11x_..."）
pred_field = "pred_yolo11l_20pct_null_images_add_rawData_batch_4_final"

# 1) 生成 patches 视图（每个检测框一个样本）
patches = dataset.to_patches(pred_field)
print(patches)  # 查看 patches 视图信息



Dataset:     test_run2_image_copy
Media type:  image
Num patches: 8320
Patch fields:
    id:                                                       fiftyone.core.fields.ObjectIdField
    sample_id:                                                fiftyone.core.fields.ObjectIdField
    filepath:                                                 fiftyone.core.fields.StringField
    tags:                                                     fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                                 fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                               fiftyone.core.fields.DateTimeField
    last_modified_at:                                         fiftyone.core.fields.DateTimeField
    pred_yolo11l_20pct_null_images_add_rawData_batch_4_final: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detection)
View stages:
    1. ToPat

In [42]:
import fiftyone as fo
from fiftyone import ViewField as F

# 假设 pred_field 是你的检测字段
pred_field = "pred_yolo11l_20pct_null_images_add_rawData_batch_4_final"

# 1) 按相对面积过滤（例如 5% ~ 50%）
# bbox_area = F("bounding_box")[2] * F("bounding_box")[3]
# filtered_rel = dataset.filter_labels(pred_field, (0.05 <= bbox_area) & (bbox_area < 0.5))

# 2) 按绝对像素面积过滤（例如 32^2 ~ 96^2 像素）
dataset.compute_metadata()  # 确保有 metadata
bbox_area_px = (
    F("$metadata.width")
    * F("bounding_box")[2]
    * F("$metadata.height")
    * F("bounding_box")[3]
)
filtered_px = dataset.filter_labels(
    pred_field, (95**2 < bbox_area_px) & (bbox_area_px < 500**2)
)

In [43]:
filtered_px

Dataset:     test_run2_image_copy
Media type:  image
Num samples: 82
Sample fields:
    id:                                                           fiftyone.core.fields.ObjectIdField
    filepath:                                                     fiftyone.core.fields.StringField
    tags:                                                         fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                                                     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                                                   fiftyone.core.fields.DateTimeField
    last_modified_at:                                             fiftyone.core.fields.DateTimeField
    pred_yolo11x_20pct_null_images_add_rawData_batch_8_final:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    pred_yolo11m_20pct_null_images_add_rawData_batch_16_final:    fiftyone.core.fields.Embedde

In [44]:
# 更新视图
session.view = filtered_px  


In [25]:
import fiftyone as fo
from PIL import Image

def ensure_det_field(ds, field):
    if field not in ds.get_field_schema():
        ds.add_sample_field(
            field,
            fo.EmbeddedDocumentField,
            embedded_doc_type=fo.Detections,
        )

def crop_with_padding(pil_img, det: fo.Detection, pad_ratio=0.2):
    W, H = pil_img.size
    x, y, w, h = det.bounding_box  # relative [0..1]
    x1, y1 = x * W, y * H
    x2, y2 = (x + w) * W, (y + h) * H

    pad_w = (x2 - x1) * pad_ratio
    pad_h = (y2 - y1) * pad_ratio

    x1 = max(0, int(x1 - pad_w))
    y1 = max(0, int(y1 - pad_h))
    x2 = min(W, int(x2 + pad_w))
    y2 = min(H, int(y2 + pad_h))

    if x2 <= x1 or y2 <= y1:
        return None

    return pil_img.crop((x1, y1, x2, y2))

def classify_and_filter_sample(
    sample,
    pred_field,
    out_field,
    clf_predict_fn,          # 你自己的分类函数：输入 list[PIL] -> list[(label, score)]
    target_label="swd",
    clf_thresh=0.8,
    pad_ratio=0.2,
    batch_size=64,
):
    dets = getattr(sample, pred_field, None)
    if dets is None or len(dets.detections) == 0:
        sample[out_field] = fo.Detections(detections=[])
        return

    with Image.open(sample.filepath).convert("RGB") as im:
        detections = dets.detections

        crops, idxs = [], []
        for i, det in enumerate(detections):
            crop = crop_with_padding(im, det, pad_ratio=pad_ratio)
            if crop is None:
                det["clf_label"] = None
                det["clf_score"] = None
                continue
            crops.append(crop)
            idxs.append(i)

        # batch 分类
        pred_map = {}
        for s in range(0, len(crops), batch_size):
            batch = crops[s:s+batch_size]
            batch_preds = clf_predict_fn(batch)  # [(label, score), ...]
            for j, (lab, sc) in enumerate(batch_preds):
                pred_map[idxs[s+j]] = (lab, float(sc))

        filtered = []
        for i, det in enumerate(detections):
            if i not in pred_map:
                continue
            lab, sc = pred_map[i]
            det["clf_label"] = lab
            det["clf_score"] = sc
            if lab == target_label and sc >= clf_thresh:
                filtered.append(det)

        sample[out_field] = fo.Detections(detections=filtered)


In [26]:
from ultralytics import YOLO

clf = YOLO("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/yolo11m-cls-best_v6.pt")
CLF_DEVICE = "cuda"

def clf_predict_fn(crops):
    # crops: list[PIL.Image]
    rs = clf.predict(source=crops, device=CLF_DEVICE, verbose=False)
    out = []
    for r in rs:
        top1 = int(r.probs.top1)
        score = float(r.probs.top1conf)
        label = r.names[top1]
        out.append((label, score))
    return out


In [27]:
# pred_field = "pred_yolo11m_20pct_null_images_add_rawData_batch_16_final"  # 举例
out_field = pred_field + "_clf"

ensure_det_field(dataset, out_field)

for sample in dataset.iter_samples(progress=True, autosave=True):
    classify_and_filter_sample(
        sample,
        pred_field=pred_field,
        out_field=out_field,
        clf_predict_fn=clf_predict_fn,
        target_label="swd",
        clf_thresh=0.5,
        pad_ratio=0.2,
        batch_size=64,
    )


 100% |█████████████████| 181/181 [38.2s elapsed, 0s remaining, 5.9 samples/s]      


### YOLO分类模型

In [ ]:
# requirements: pip install ultralytics pandas tabulate
from ultralytics import YOLO
import os
from pathlib import Path
import pandas as pd
from tabulate import tabulate
from tqdm import tqdm
import torch

# ────────────────────────────────────────────────
#               修改这三个地方即可
# ────────────────────────────────────────────────
# MODEL_PATH = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/data_split_0.6_0.2_0.2_yolo11m-cls.pt_8.pt"      # ← 你的模型路径
# MODEL_PATH = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/yolo11m-cls-best_v6.pt"      # ← 你的模型路径
MODEL_PATH = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/yolo11m-cls-best_v7.pt"      # ← 你的模型路径
IMAGE_FOLDER = "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/patch_data/swd"                         # ← 图片所在的文件夹
CONF_THRESHOLD = 0.30                                   # ← 低于这个置信度就标为 low_conf
# ────────────────────────────────────────────────

# 加载模型
model = YOLO(MODEL_PATH)
model.to('cuda' if torch.cuda.is_available() else 'cpu')
print(f"模型加载完成：{MODEL_PATH}")
print(f"类别顺序：{model.names}\n")   # 应该显示 ['nonSWD', 'SWD'] 或相反

# 支持的图片后缀
IMG_EXT = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

# 收集所有图片
image_paths = []
for root, _, files in os.walk(IMAGE_FOLDER):
    for file in files:
        if file.lower().endswith(IMG_EXT):
            image_paths.append(os.path.join(root, file))

print(f"找到 {len(image_paths)} 张图片\n")

# 开始推理
results_list = []

for img_path in tqdm(image_paths, desc="推理中"):
    # 推理
    results = model(img_path, verbose=False)
    
    # YOLO分类的 results[0].probs 是 Top-5 格式
    probs = results[0].probs.data.cpu().numpy()   # shape: (n_classes,)
    top1_idx = int(probs.argmax())
    top1_conf = float(probs[top1_idx])
    top1_class = model.names[top1_idx]
    
    # 第二大概率（用于观察区分度）
    probs_sorted = probs.argsort()[::-1]
    second_idx = probs_sorted[1]
    second_conf = float(probs[second_idx])
    
    status = "OK"
    if top1_conf < CONF_THRESHOLD:
        status = "low_conf"
    elif top1_class not in ["SWD", "nonSWD"]:
        status = "unknown_class"
    
    results_list.append({
        "filename": Path(img_path).name,
        "path": str(Path(img_path).relative_to(IMAGE_FOLDER)),
        "pred": top1_class,
        "conf": f"{top1_conf:.4f}",
        "second": model.names[second_idx],
        "second_conf": f"{second_conf:.4f}",
        "status": status
    })

# ── 转为 DataFrame ───────────────────────────────────────
df = pd.DataFrame(results_list)

# 按置信度从高到低排序（可选）
df["conf_float"] = df["conf"].astype(float)
df = df.sort_values("conf_float", ascending=False).drop(columns="conf_float")

# ── 打印美观表格 ─────────────────────────────────────────
print("\n" + "="*80)
print("分类结果汇总")
print("="*80)

print(tabulate(
    df[["filename", "pred", "conf", "second", "status"]],
    headers="keys",
    tablefmt="simple_outline",
    showindex=False,
    floatfmt=".4f"
))

# 统计信息
print("\n统计：")
print(df["pred"].value_counts())
print("\n低置信度样本数：", (df["conf"].astype(float) < CONF_THRESHOLD).sum())
print("异常样本数：", len(df[~df["status"].isin(["OK"])]))

# 可选：保存到 csv
# df.to_csv("classification_result.csv", index=False, encoding="utf-8-sig")
print("\n结果已保存到：classification_result.csv")
df

模型加载完成：/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/yolo11m-cls-best_v7.pt
类别顺序：{0: 'mayswd', 1: 'others', 2: 'swd'}

找到 8320 张图片



推理中: 100%|██████████| 8320/8320 [00:17<00:00, 484.78it/s]



分类结果汇总
┌────────────┬────────┬────────┬──────────┬───────────────┐
│ filename   │ pred   │   conf │ second   │ status        │
├────────────┼────────┼────────┼──────────┼───────────────┤
│ 006055.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 006412.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 002742.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 007499.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 000735.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 002604.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 002547.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 001705.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 002077.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 003252.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 001191.jpg │ swd    │ 1.0000 │ others   │ unknown_class │
│ 000187.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 000313.jpg │ swd    │ 1.0000 │ mayswd   │ unknown_class │
│ 001294.jpg │ swd    │ 1.0000 │

In [29]:
df

,filename,path,pred,conf,second,second_conf,status
5,006055.jpg,006055.jpg,swd,1.0000,mayswd,0.0000,unknown_class
10,006412.jpg,006412.jpg,swd,1.0000,others,0.0000,unknown_class
8288,002742.jpg,002742.jpg,swd,1.0000,others,0.0000,unknown_class
1831,007499.jpg,007499.jpg,swd,1.0000,others,0.0000,unknown_class
1837,000735.jpg,000735.jpg,swd,1.0000,mayswd,0.0000,unknown_class
...,...,...,...,...,...,...,...
2402,007118.jpg,007118.jpg,others,0.3940,mayswd,0.3523,unknown_class
7308,004326.jpg,004326.jpg,others,0.3910,mayswd,0.3160,unknown_class
351,006738.jpg,006738.jpg,swd,0.3884,mayswd,0.3167,unknown_class
6135,001363.jpg,001363.jpg,others,0.3769,swd,0.3119,unknown_class


In [30]:
import shutil
from pathlib import Path

# 复制图片到对应的文件夹
for _, row in df.iterrows():
    src_path = Path(IMAGE_FOLDER) / row["path"]
    dst_folder = Path(IMAGE_FOLDER).parent /"classified_images" / row["pred"]
    
    os.makedirs(dst_folder, exist_ok=True)
    dst_path = dst_folder / row["filename"]
    
    # 复制文件
    try:
        shutil.copy(src_path, dst_path)
    except Exception as e:
        print(f"复制失败: {src_path} -> {dst_path}, 错误: {e}")